# 02 — Feature Extraction

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("hit-predictor").getOrCreate()

In [ ]:
billboard = pd.read_csv("../data/raw/hot100.csv")
billboard = billboard.rename(columns={
    "Song": "title",
    "Artist": "artist",
    "Peak Position": "peak_pos",
    "Weeks in Charts": "weeks_on_chart"
})
billboard["label"] = (billboard["peak_pos"] <= 10).astype(int)
billboard = billboard[["title", "artist", "label"]].drop_duplicates(subset=["title", "artist"])

spotify = pd.read_csv("../data/raw/dataset.csv")
spotify = spotify.rename(columns={"track_name": "title", "artists": "artist"})

In [ ]:
for df in [billboard, spotify]:
    df["title"] = df["title"].str.lower().str.strip()
    df["artist"] = df["artist"].str.lower().str.strip()

merged = billboard.merge(spotify, on=["title", "artist"], how="inner")
print(merged.shape)

In [ ]:
FEATURE_COLS = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

merged = merged[FEATURE_COLS + ["title", "artist", "label"]].dropna()

In [ ]:
# merged.to_csv("../data/processed/features.csv", index=False)